# RAG (Step by Step Implementation)

- Introduction
    - Large Language Models (LLMs) are powerful but have limitations:
        - They may generate hallucinated (incorrect) answers
        - They do not have access to external or private data
        - Their knowledge is limited to training data

    To overcome these challenges, we use Retrieval-Augmented Generation (RAG).

+ What is RAG?
    - RAG is a technique that combines:
        - Retrieval -> Fetch relevant information from external data
        - Generation -> Use an LLM to generate answers based on retrieved context

- RAG Pipeline:
    1. User Query
    2. Convert query into embedding (vector)
    3. Retrieve relevant chunks from a Vector Database (FAISS)
    4. Pass retrieved context + query to LLM
    5. Generate final answer


## Setup

In [ ]:
## Installing dependencies

# !pip install langchain-groq sentence-transformers faiss-cpu python-dotenv

In [2]:
## Setting up the langchain-groq client and response generation function
# Here we are using langchain-groq to interact with the Groq API. The `generate_response` function takes a prompt and optional parameters for temperature and max tokens, 
# and returns the model's response or an error message if the API call fails.


from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
import json
from dotenv import load_dotenv

load_dotenv()   # Loading the env variable


def generate_response(prompt, temperature=0.7, max_tokens=500):
    """
    Sends a prompt to Groq and returns the text response.

    Parameters:
        prompt (str): The full prompt string to send.
        temperature (float): Controls randomness. 0 = deterministic, 1 = creative.
        max_tokens (int): Max length of the response.

    Returns:
        str: The model's text response, or an error message.
    """
    try:
        llm = ChatGroq(
            model="llama-3.1-8b-instant",
            temperature=temperature,
            max_tokens=max_tokens,
        )
        response = llm.invoke([HumanMessage(content=prompt)])
        return response.content

    except Exception as e:
        return f"[API Error] {str(e)}"


d:\Work\Learning\Topics_by_Abhay_Sir\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
## Testing the response generation function with a simple prompt

test_prompt = "What is Retrieval Augmented Generation?"
response = generate_response(test_prompt)
print(response)

Retrieval Augmented Generation (RAG) is a technique in natural language processing (NLP) that combines the strengths of two models: a retriever and a generator. 

The retriever is a model that searches a large database or knowledge base to find relevant information related to a given input or prompt. The generator is a model that uses the information retrieved by the retriever to generate a response, such as text or a summary.

Here's a step-by-step overview of how RAG works:

1. **Retriever:** The input prompt is passed to the retriever, which searches the knowledge base to find relevant documents or passages that match the prompt.
2. **Ranking:** The retriever ranks the retrieved documents based on their relevance to the prompt. This ranking is typically done using a scoring function that evaluates the similarity between the input and the document.
3. **Generator:** The top-ranked document(s) are passed to the generator, which uses the information in the document to generate a respon

In [ ]:
## Setting up the embedding model
## Here, we will use the `sentence-transformers` library to create an embedding model. The `create_embedding` function takes a text input and returns its vector representation using the specified model.

from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

def create_embedding(text):
    """
    Converts text into a vector embedding.

    Parameters:
        text (str): The text to embed.

    Returns:
        list: The embedding vector as a list of floats.
    """
    return embedding_model.encode(text)

# (Wy embeddings ?
# Because it convert text into vectors for semantic similarity search and retrieval tasks.)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4531.62it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
## Testing embeddings

sentences = [
    "RAG improves accuracy",
    "Dogs are animals",
    "RAG helps reduce hallucination"
]

embeddings = create_embedding(sentences)

print("Embedding shape:", embeddings.shape)

Embedding shape: (3, 384)


In [ ]:
embeddings[0]  # Print the first embedding vector

array([-4.07009460e-02,  5.93289994e-02,  8.15168843e-02, -5.53964786e-02,
       -1.17434166e-01, -3.10564041e-02,  1.55403288e-02,  3.78301553e-02,
       -5.37311882e-02, -2.70676352e-02, -2.38407832e-02,  1.64680909e-02,
        1.38016632e-02, -2.22222004e-02, -8.01788121e-02,  2.12712083e-02,
        7.19759539e-02,  1.86225623e-01, -6.17407858e-02, -7.32362643e-02,
       -1.02868959e-01,  1.00873401e-02,  4.68627699e-02,  5.25569543e-03,
        2.43153721e-02, -3.64282634e-03, -6.67842180e-02,  5.24282316e-03,
        9.60860625e-02, -8.77793059e-02, -2.27026758e-03,  7.15932995e-02,
       -3.49835753e-02, -6.64551407e-02, -8.34993646e-02, -1.15731414e-02,
       -5.75707890e-02,  8.04994702e-02, -9.13321599e-03,  2.20366064e-02,
        4.56162691e-02, -8.60973913e-03,  3.52232382e-02, -4.01451550e-02,
        1.46522140e-02,  6.28121719e-02, -1.75774992e-02,  5.34058884e-02,
        1.53937824e-02, -2.19031218e-02, -4.72565778e-02,  6.29423559e-03,
        6.26454428e-02, -

In [ ]:
## Setting up FAISS

import faiss
import numpy as np

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("Number of vectors in index:", index.ntotal)


# (Why FAISS ?
# Because it is a library for efficient similarity search (like efficient nearest neighbor search in vector space) and clustering of dense vectors, making it ideal for
# handling large-scale embedding data in RAG systems.)

Number of vectors in index: 3


In [ ]:
## Testing similarity search

query = "How does RAG help?"
query_embedding = embedding_model.encode([query])
distances, indices = index.search(np.array(query_embedding), k=2)   # k=2 for top 2 matches

print("Top matches:", indices)


# (Here we are using FAISS to perform a similarity search. We encode a query into an embedding and search for the closest vectors in our index, which contains the embeddings of 
# our sentences. The output will show the indices of the top matches based on cosine similarity.)

Top matches: [[2 0]]


## Load Document

In [43]:
def load_text_file(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        text = file.read()
    return text

document = load_text_file("data/sample.txt")

print("Document loaded successfully.")
print(document[:300])  # preview first 300 characters
print("\nDocument length:", len(document))

Document loaded successfully.
Retrieval-Augmented Generation (RAG) is a technique that combines retrieval and generation.
It helps large language models access external knowledge.
RAG reduces hallucination by grounding responses in retrieved context.
It uses embeddings to perform semantic search.
Vector databases like FAISS help

Document length: 8518


In [44]:
## Basc text cleaning

def clean_text(text):
    text = text.replace("\n", " ")
    text = text.strip()
    return text

cleaned_text = clean_text(document)

print(cleaned_text[:300])
print("\nCleaned document length:", len(cleaned_text))

Retrieval-Augmented Generation (RAG) is a technique that combines retrieval and generation. It helps large language models access external knowledge. RAG reduces hallucination by grounding responses in retrieved context. It uses embeddings to perform semantic search. Vector databases like FAISS help

Cleaned document length: 8518


## Basic Chunking

```
Goal of this step -
    Split document into chunks
    Prepare chunks for embeddings

Why Chunking?
    LLMs cannot take very large text at once, so we:
    Break document -> smaller pieces (chunks)
    Retrieve only relevant chunks during query

    Bad chunking = bad retrieval = bad answers

```

In [45]:
## Converting text into langchain document format so that we can use it with langchain's retrieval and generation capabilities. 

from langchain_core.documents import Document

documents = [Document(page_content=cleaned_text)]
documents[0]    # Print the first document to verify the format

Document(metadata={}, page_content='Retrieval-Augmented Generation (RAG) is a technique that combines retrieval and generation. It helps large language models access external knowledge. RAG reduces hallucination by grounding responses in retrieved context. It uses embeddings to perform semantic search. Vector databases like FAISS help in efficient retrieval of relevant chunks.  What is Retrieval-Augmented Generation? Retrieval-Augmented Generation (RAG) is the process of optimizing the output of a large language model, so it references an authoritative knowledge base outside of its training data sources before generating a response. Large Language Models (LLMs) are trained on vast volumes of data and use billions of parameters to generate original output for tasks like answering questions, translating languages, and completing sentences. RAG extends the already powerful capabilities of LLMs to specific domains or an organization\'s internal knowledge base, all without the need to retra

In [35]:
# !pip install -U langchain-text-splitters

In [ ]:
## Doing basic chunking using CharacterTextSplitter

from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    chunk_size=100,   # characters per chunk. This basically splits the text into very small pieces for demonstration. In real scenarios, you would use a larger chunk size (e.g., 500-1000 characters).
    chunk_overlap=0   # no overlap (basic version)
)

chunks = text_splitter.split_documents(documents)
# chunks[0]  # Print the first chunk to verify the splitting
chunks

# (CharacterTextSplitter does not split purely by length. It tires to split based on separator first (like \n\n (double newline)).
# So it treats entire text as ONE chunk) 

[Document(metadata={}, page_content='Retrieval-Augmented Generation (RAG) is a technique that combines retrieval and generation. It helps large language models access external knowledge. RAG reduces hallucination by grounding responses in retrieved context. It uses embeddings to perform semantic search. Vector databases like FAISS help in efficient retrieval of relevant chunks.  What is Retrieval-Augmented Generation? Retrieval-Augmented Generation (RAG) is the process of optimizing the output of a large language model, so it references an authoritative knowledge base outside of its training data sources before generating a response. Large Language Models (LLMs) are trained on vast volumes of data and use billions of parameters to generate original output for tasks like answering questions, translating languages, and completing sentences. RAG extends the already powerful capabilities of LLMs to specific domains or an organization\'s internal knowledge base, all without the need to retr

In [ ]:
## Doing basic chunking using RecursiveCharacterTextSplitter

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,   # characters per chunk. This basically splits the text into very small pieces for demonstration. In real scenarios, you would use a larger chunk size (e.g., 500-1000 characters).
    chunk_overlap=0   # no overlap (basic version)
)

chunks = text_splitter.split_documents(documents)
# chunks[0]  # Print the first chunk to verify the splitting
chunks

# (RecursiveCharacterTextSplitter is more flexible and tries to split the text based on a hierarchy of separators (like \n\n, \n, space) while also respecting the chunk size.
# So it creates multiple chunks based on the specified chunk size and the structure of the text.)

[Document(metadata={}, page_content='Retrieval-Augmented Generation (RAG) is a technique that combines retrieval and generation. It helps large language models access external knowledge. RAG reduces hallucination by grounding responses'),
 Document(metadata={}, page_content='in retrieved context. It uses embeddings to perform semantic search. Vector databases like FAISS help in efficient retrieval of relevant chunks.  What is Retrieval-Augmented Generation?'),
 Document(metadata={}, page_content='Retrieval-Augmented Generation (RAG) is the process of optimizing the output of a large language model, so it references an authoritative knowledge base outside of its training data sources before'),
 Document(metadata={}, page_content='generating a response. Large Language Models (LLMs) are trained on vast volumes of data and use billions of parameters to generate original output for tasks like answering questions, translating'),
 Document(metadata={}, page_content="languages, and completing 

In [53]:
## Inspecting chunks

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks[:3]):
    print(f"\nChunk {i+1}:\n", chunk.page_content)

Number of chunks: 44

Chunk 1:
 Retrieval-Augmented Generation (RAG) is a technique that combines retrieval and generation. It helps large language models access external knowledge. RAG reduces hallucination by grounding responses

Chunk 2:
 in retrieved context. It uses embeddings to perform semantic search. Vector databases like FAISS help in efficient retrieval of relevant chunks.  What is Retrieval-Augmented Generation?

Chunk 3:
 Retrieval-Augmented Generation (RAG) is the process of optimizing the output of a large language model, so it references an authoritative knowledge base outside of its training data sources before


In [54]:
## Checking chubk size

for i, chunk in enumerate(chunks[:5]):
    print(f"Chunk {i+1} length:", len(chunk.page_content))

Chunk 1 length: 198
Chunk 2 length: 185
Chunk 3 length: 196
Chunk 4 length: 194
Chunk 5 length: 199


## Embeddings + Vector Store

```
Convert chunks -> embeddings
Store embeddings in FAISS
Perform similarity search

Each chunk:
    "RAG improves accuracy..."
becomes -
    [0.12, -0.45, 0.88, ...]  (vector)
```

In [58]:
# ! pip install langchain-huggingface

In [ ]:
## Set up embeddings for chunks

from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

# (Have already done this previously but we are doing it again here for the sake of continuity in the notebook. This sets up the embedding model that we will use to convert our text chunks into vector representations for retrieval.)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3316.19it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [62]:
# !pip install langchain_community

In [63]:
## Create FAISS vector store

from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    chunks,
    embedding_model
)

In [ ]:
## Performing similarity search with the vector store

query = "How does RAG reduce hallucination?"

results = vectorstore.similarity_search(query, k=4)

## Looking at retrieved chunks

for i, doc in enumerate(results):
    print(f"\nResult {i+1}:\n", doc.page_content)


# (What just happened?
# Each chunk -> converted into embedding
# Stored in FAISS
# Query -> converted into embedding
# FAISS -> finds closest chunks)
# So unitl now we have done the retrieval part of RAG. We have retrieved relevant chunks based on the query. Next, we will use these retrieved chunks to generate a response using the llm.


Result 1:
 Retrieval-Augmented Generation (RAG) is a technique that combines retrieval and generation. It helps large language models access external knowledge. RAG reduces hallucination by grounding responses

Result 2:
 relevance to maximize the quality of the RAG payload.

Result 3:
 for your needs, it is challenging to maintain relevancy. RAG allows developers to provide the latest research, statistics, or news to the generative models. They can use RAG to connect the LLM

Result 4:
 languages, and completing sentences. RAG extends the already powerful capabilities of LLMs to specific domains or an organization's internal knowledge base, all without the need to retrain the model.


## Build Full RAG Pipeline (Retrieval + Generation)

```
Take user query
Retrieve relevant chunks
Pass them to LLM
Generate final answer

Until now:
    You had retrieval working
Now:
    Retrieval + LLM = RAG
```

In [67]:
## Defining retrieval function

def retrieve_chunks(query, k=3):
    results = vectorstore.similarity_search(query, k=k)
    return results

In [68]:
## Creating context from retrieved chunks because we want to feed this context to the llm for response generation. The `build_context` function takes the retrieved documents and concatenates their content into a single string that can be used as context for the language model.

def build_context(docs):
    context = "\n\n".join([doc.page_content for doc in docs])
    return context

In [69]:
## Basic RAG prompt

def create_prompt(query, context):
    prompt = f"""
Answer the question based on the context below:

Context:
{context}

Question:
{query}
"""
    return prompt

In [70]:
## Full RAG pipeline

def rag_pipeline(query, k=3):
    # Step 1: Retrieve
    docs = retrieve_chunks(query, k)
    
    # Step 2: Build context
    context = build_context(docs)
    
    # Step 3: Create prompt
    prompt = create_prompt(query, context)
    
    # Step 4: Generate answer
    answer = generate_response(prompt)
    
    return answer, docs

In [ ]:
## Testing the RAG pipeline

query = "How does RAG reduce hallucination?"

answer, docs = rag_pipeline(query)

print("Answer:\n", answer)

# (Here we are testing the entire RAG pipeline. We input a query, the system retrieves relevant chunks, builds a context, creates a prompt, and generates an answer using the language model. The retrieved chunks are also returned for inspection.
# But here the respose quality is not good because we have used a very small chunk size and only 3 chunks for retrieval. In real scenarios, you would use larger chunk sizes and more retrieved chunks to provide richer context to the model, which can lead to better responses.

# So will further improve few parts like prompt and chunking with overlap to see how it affects the response quality.)

Answer:
 According to the context, RAG (Retrieval-Augmented Generation) reduces hallucination by grounding responses relevance to maximize the quality of the RAG payload.


In [72]:
## Inspecting retrieved documents

print("\nRetrieved Chunks:\n")

for i, doc in enumerate(docs):
    print(f"\nChunk {i+1}:\n", doc.page_content)


Retrieved Chunks:


Chunk 1:
 Retrieval-Augmented Generation (RAG) is a technique that combines retrieval and generation. It helps large language models access external knowledge. RAG reduces hallucination by grounding responses

Chunk 2:
 relevance to maximize the quality of the RAG payload.

Chunk 3:
 for your needs, it is challenging to maintain relevancy. RAG allows developers to provide the latest research, statistics, or news to the generative models. They can use RAG to connect the LLM


In [73]:
query = "Who invented RAG?"

answer, docs = rag_pipeline(query)

print("Answer:\n", answer)


Answer:
 The text doesn't explicitly mention who invented RAG. However, based on the given context, it seems that RAG is related to the capabilities of Large Language Models (LLMs). 

RAG is explained as extending the capabilities of LLMs to specific domains or an organization's internal knowledge base. It's also mentioned that it doesn't require retraining the model. However, the originator or the name of the person who developed RAG is not mentioned in the provided text.


## Improvements

### Prompt Engineering

```
Force LLM to stick to retrieved context
Reduce hallucination
Make answers trustworthy

Your current prompt:
    Answer the question based on the context below
Issue:
    LLM can still use its own knowledge 
    Can generate incorrect info 
Solution: 
    Strong RAG Prompt
We constrain the LLM
```

In [74]:
## Update prompt function

def create_prompt(query, context):
    prompt = f"""
You are a helpful assistant.

Use ONLY the provided context to answer the question.
Do NOT use any external knowledge.

If the answer is not present in the context, say:
"I don't know based on the provided context."

Context:
{context}

Question:
{query}

Answer:
"""
    return prompt

In [ ]:
## Again testing (with knows asnwer whose answer is present in the document)

query = "What is RAG?"

answer, docs = rag_pipeline(query)

print("Answer:\n", answer)

# (This has given answer strictly from the documents. The updated prompt explicitly instructs the model to rely solely on the provided context and to admit when the answer is not available, which helps in reducing hallucination and improving the accuracy of the response.)


Answer:
 RAG (short for Retrieval-Augmented Generation) extends the capabilities of LLMs to specific domains or an organization's internal knowledge base, allowing for solutions without the need to retrain the model.


In [ ]:
## ## Again testing (this time with a question whose answer is NOT present in the document to see if the model correctly says "I don't know based on the provided context.")

query = "Who invented RAG?"

answer, docs = rag_pipeline(query)

print("Answer:\n", answer)

# (As we can see that it has cleary stated that the answer is not present in the document, which indicates that the model is correctly following the instructions in 
# the prompt and not hallucinating an answer based on its pre-trained knowledge. )


## So we have now have : 
    # Controlled LLM behavior
    # Reduced hallucination
    # Reliable RAG responses

Answer:
 I don't know based on the provided context.


### Chunking

```
Fix context breaking issue
Improve retrieval quality
Compare before vs after chunking

Problem with Current Chunking
Right now we have used:
    chunk_size=100
    chunk_overlap=0

Issue:
Example:
    Chunk 1: "RAG improves accuracy by retrieving relevant"
    Chunk 2: "information from external sources"

    Meaning is split
    Retrieval may miss full context

Solution: Overlapping Chunks
We introduce overlap
Idea:
    Next chunk repeats part of previous chunk
```

In [77]:
## Update chunking code with overlap

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,      # bigger chunk
    chunk_overlap=60    # overlap added
)

improved_chunks = text_splitter.split_documents(documents)

In [ ]:
## Inspecting new chuks with overlap

print("Number of improved chunks:", len(improved_chunks))

for i, chunk in enumerate(improved_chunks[:3]):
    print(f"\nChunk {i+1}:\n", chunk.page_content)

# (Here we can see that the chunks now have some overlap, which means that important information that might be split between two chunks can still be captured in both chunks. 
# This can help improve retrieval performance and the quality of generated responses, especially when key information is located at the boundaries of chunks.)

Number of improved chunks: 36

Chunk 1:
 Retrieval-Augmented Generation (RAG) is a technique that combines retrieval and generation. It helps large language models access external knowledge. RAG reduces hallucination by grounding responses in retrieved context. It uses embeddings to perform semantic search. Vector databases like FAISS help

Chunk 2:
 perform semantic search. Vector databases like FAISS help in efficient retrieval of relevant chunks.  What is Retrieval-Augmented Generation? Retrieval-Augmented Generation (RAG) is the process of optimizing the output of a large language model, so it references an authoritative knowledge base

Chunk 3:
 model, so it references an authoritative knowledge base outside of its training data sources before generating a response. Large Language Models (LLMs) are trained on vast volumes of data and use billions of parameters to generate original output for tasks like answering questions, translating


In [79]:
## Rebuilding vector store with improved chunks

from langchain_community.vectorstores import FAISS

improved_vectorstore = FAISS.from_documents(
    improved_chunks,
    embedding_model
)

In [80]:
## Updating retrival function to use improved vector store

def retrieve_chunks(query, k=3):
    results = improved_vectorstore.similarity_search(query, k=k)
    return results

In [ ]:
## Runiing same query with improved chunks and vector store to compare results

query = "How does RAG reduce hallucination?"

answer, docs = rag_pipeline(query)

print("Answer:\n", answer)

# (Compared to before:
    # More complete answers
    # Better context flow
    # Less missing information)

# So initially I used basic chunking, but it caused context fragmentation. I improved it using overlapping chunks, which significantly enhanced retrieval quality and answer completeness.

# So now we have a more robust RAG pipeline with better chunking and a prompt that reduces hallucination, leading to more accurate and reliable responses from the language model.
    # Improved chunking strategy
    # Better retrieval
    # Stronger RAG system

Answer:
 RAG reduces hallucination by grounding responses in retrieved context.


### Adding Citations

```
Show where the answer comes from
Make output transparent & explainable
Improve trust in RAG system

Right now your output is:
    Answer: RAG reduces hallucination by...

Problem:
    No idea which chunk was used

After this step:
    Answer: RAG reduces hallucination by grounding responses...
    Sources:
    - Chunk 1
    - Chunk 3

This is explainable AI
```

In [ ]:
## Adding metadata to chunks before creating FAISS vector store
## Updating chunk creation to include metadata (like chunk number) which can help in debugging and understanding which part of the document is being retrieved.

for i, chunk in enumerate(improved_chunks):
    chunk.metadata["chunk_id"] = i


In [89]:
## Recreate Vector Store

from langchain_community.vectorstores import FAISS

improved_vectorstore = FAISS.from_documents(
    improved_chunks,
    embedding_model
)

In [90]:
## Update Context Builder
# We now keep both:
# text
# chunk IDs

def build_context_with_sources(docs):
    context = ""
    sources = []
    
    for doc in docs:
        context += doc.page_content + "\n\n"
        sources.append(doc.metadata.get("chunk_id", "N/A"))
    
    return context, sources

In [91]:
## Upding RAG pipeline to use new context builder

def rag_pipeline(query, k=3):
    # Step 1: Retrieve
    docs = retrieve_chunks(query, k)
    
    # Step 2: Build context + sources
    context, sources = build_context_with_sources(docs)
    
    # Step 3: Create prompt
    prompt = create_prompt(query, context)
    
    # Step 4: Generate answer
    answer = generate_response(prompt)
    
    return answer, sources, docs

In [92]:
## Printing answer with source information

query = "How does RAG reduce hallucination?"

answer, sources, docs = rag_pipeline(query)

print("Answer:\n", answer)

print("\nSources:")
for s in sources:
    print(f"- Chunk {s}")

# (So now this gives Answer + supporting chunks
# Builds trust
# Helps debug retrieval
# Makes system explainable)

Answer:
 RAG reduces hallucination by grounding responses in retrieved context.

Sources:
- Chunk 3
- Chunk 14
- Chunk 0


## **Now next we will proceed to Notebook 2 for Advance RAG cvering topics like Chunking strategies, RAG patterns, Hybrid search and rerenaking.**